# Lab 4: Short-Term Memory & Streaming

**Difficulty: Intermediate | ~40 min | Requires Lab 3**

## Step 1 — Install the required modules

In [ ]:
# One command installs all required modules (versions pinned for reproducibility)
!pip install "langchain==1.2.15" "langchain-core==1.2.28" "langchain-openai==1.1.12" "python-dotenv==1.2.2" "pydantic==2.13.4"

## Step 2 — Load your API key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENROUTER_API_KEY"):
    raise SystemExit("No OPENROUTER_API_KEY found. Add it to .env and restart the kernel.")

## Step 3 — Create the model

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    temperature=0,
)

## Step 4 — Build the prompt and the in-memory history store

The prompt is a template: a fixed **system** instruction, a `history` slot that
LangChain fills with the past turns, and the current `{input}`. The store is a
plain dict mapping a session ID to a `ChatMessageHistory` object that holds that
session's messages. `get_session_history` is the callback `RunnableWithMessageHistory`
will call to load (or create) the history for whatever session is active.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder("history"),
    ("human", "{input}"),
])

store = {}

def get_session_history(session_id):
    return store.setdefault(session_id, InMemoryChatMessageHistory())

## Step 5 — Wrap the chain with RunnableWithMessageHistory

`RunnableWithMessageHistory` is a wrapper around the plain `prompt | model`
chain. On every call it: (1) loads the session's history via the callback,
(2) injects it into the prompt's `history` slot, and (3) after the model
replies, appends the new question and answer back to the store — so the next
turn sees them. The two `*_messages_key` arguments tell the wrapper which
prompt variables hold the incoming text and the history.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory

chain = prompt | model
chat = RunnableWithMessageHistory(
    chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

## Step 6 — Chat across turns (memory works)

Both calls use `session_id="ada"`, so they share one history. The first turn
states a fact; the second turn asks about it. Because the wrapper replayed the
first exchange into the prompt, the model answers from context — proof the
conversation is now short-term memory, not a stateless prompt.

In [ ]:
response = chat.invoke(
    {"input": "Hi, my name is Ada."},
    config={"configurable": {"session_id": "ada"}},
)
print(response.content)

response = chat.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "ada"}},
)
print(response.content)

print(f"History stored for 'ada': {len(store['ada'].messages)} messages")


## Step 7 — Each session has its own memory

Now the same question through a different `session_id`. The wrapper looks up a
different history (an empty one), so the model has no idea about Ada — the
memory is scoped per session, exactly how a chat app keeps one context per
conversation. Finally we list the store's keys to show the two conversations
live side by side.

In [ ]:
response = chat.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "new-user"}},
)
print(response.content)

print(f"\nSessions in the store: {sorted(store.keys())}")

print("\nWhat the ada session remembered:")
for message in store["ada"].messages:
    print(f"  {type(message).__name__}: {message.content[:60]}")

## Step 8 — Stream a response token by token

`model.stream(...)` returns an iterator of **chunks** instead of one finished
answer. Each chunk is an `AIMessageChunk` whose `.content` holds the tokens
produced so far, so printing `end=""` reassembles the text as it arrives — the
mechanism behind the typewriter effect of ChatGPT-like UIs. We also count the
chunks and remember the first one's type to make the streaming explicit.

In [ ]:
first_chunk = None
chunk_count = 0
for chunk in model.stream("Write a short two-line poem about coffee."):
    if first_chunk is None:
        first_chunk = chunk
    chunk_count += 1
    print(chunk.content, end="", flush=True)

print(f"\n\nStreamed in {chunk_count} chunks; first chunk type: {type(first_chunk).__name__}")

## Step 9 — Stream with memory

Streaming and memory compose: we stream through the *wrapped* `chat` object,
still passing the `session_id` that already knows Ada. The question is answered
from history — but the tokens arrive as a stream, exactly as a production chat
UI would render them.

In [ ]:
for chunk in chat.stream(
    {"input": "In one word, what is my name?"},
    config={"configurable": {"session_id": "ada"}},
):
    print(chunk.content, end="", flush=True)
print()

## Optional Exercise — Cap the memory window

Unbounded history grows until it blows past the model's context limit. Add a
**trimmer** that keeps only the most recent messages before the prompt sees
them. Build a second chain exactly like Steps 4–5 but with
`RunnablePassthrough.assign(history=itemgetter("history") | trimmer)` inserted
before `prompt`, then hold three turns of conversation and confirm the model no
longer recalls the very first turn.

In [ ]:
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter


def count_words(messages):
    return sum(len(m.content.split()) for m in messages)


trimmer = trim_messages(
    max_tokens=25,
    strategy="last",
    token_counter=count_words,
    include_system=True,
    allow_partial=False,
    start_on="human",
)

bounded_chain = (
    RunnablePassthrough.assign(history=itemgetter("history") | trimmer)
    | prompt
    | model
)
bounded_chat = RunnableWithMessageHistory(
    bounded_chain,
    get_session_history=get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

bounded_chat.invoke(
    {"input": "My favorite city is Oslo. Remember it."},
    config={"configurable": {"session_id": "windowed"}},
)
bounded_chat.invoke(
    {"input": "I also love winter hiking."},
    config={"configurable": {"session_id": "windowed"}},
)
response = bounded_chat.invoke(
    {"input": "What is my favorite city?"},
    config={"configurable": {"session_id": "windowed"}},
)
print(response.content)